In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# Least-to-Most Prompting

## chain-of-thought (one-shot)

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# Prompt
template = """
Q: Elsa有5个苹果。Anna比Elsa多两个苹果。她们一共有多少个苹果？
A: Anna比Elsa多了 2 个苹果。所以Anna有 2 + 5 = 7 个苹果。Elsa和Anna一共有 5 + 7 = 12 个苹果。答案是12。

Q：{question}
A：
"""

prompt = ChatPromptTemplate.from_messages(
    ("system", template)
)

# LLM
llm = ChatOpenAI(temperature=0)

# Output Parser
output_parser = StrOutputParser()

# Chain
cot_chain = prompt | llm | output_parser

In [3]:
question = "Albert 想知道他一天能吃多少披萨。他买了 2 个大披萨和 2 个小披萨。一个大披萨有 16 片，一个小披萨有 8 片。如果他全部吃掉，那天他吃了多少块？"

In [4]:
cot_chain.invoke(question)

'Albert买了2个大披萨，每个大披萨有16片，所以总共有2 * 16 = 32片大披萨。\n他还买了2个小披萨，每个小披萨有8片，所以总共有2 * 8 = 16片小披萨。\n如果他全部吃掉，那么他一共吃了32 + 16 = 48片披萨。所以他那天吃了48片披萨。'

## least-to-most (one-shot)

### 初始请求

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# Prompt
first_template = """
Q: Elsa有5个苹果。Anna比Elsa多两个苹果。她们一共有多少个苹果？
A: 让我们把这个问题分解一下：1.Anna有多少个苹果？2. Elsa和Anna一共有多少个苹果？
1. Anna比Elsa多 2 个苹果。所以Anna有 2 + 5 = 7 个苹果。
2. Elsa和Anna一共有 5 + 7 = 12 个苹果。

Q：{question}
A：让我们把这个问题分解一下：
"""

first_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", first_template)
    ]
)

# LLM
llm = ChatOpenAI(temperature=0)

# Output Parser
output_parser = StrOutputParser()

# Chain
l2m_first_chain = first_prompt | llm | output_parser

In [7]:
question

'Albert 想知道他一天能吃多少披萨。他买了 2 个大披萨和 2 个小披萨。一个大披萨有 16 片，一个小披萨有 8 片。如果他全部吃掉，那天他吃了多少块？'

In [6]:
first_reply = l2m_first_chain.invoke({"question": question})
first_reply

'1. 两个大披萨一共有 2 * 16 = 32 片。\n2. 两个小披萨一共有 2 * 8 = 16 片。\n3. Albert 一共吃了 32 + 16 = 48 片披萨。'

In [8]:
print(first_prompt.invoke({"question": question}).messages[0].content)


Q: Elsa有5个苹果。Anna比Elsa多两个苹果。她们一共有多少个苹果？
A: 让我们把这个问题分解一下：1.Anna有多少个苹果？2. Elsa和Anna一共有多少个苹果？
1. Anna比Elsa多 2 个苹果。所以Anna有 2 + 5 = 7 个苹果。
2. Elsa和Anna一共有 5 + 7 = 12 个苹果。

Q：Albert 想知道他一天能吃多少披萨。他买了 2 个大披萨和 2 个小披萨。一个大披萨有 16 片，一个小披萨有 8 片。如果他全部吃掉，那天他吃了多少块？
A：让我们把这个问题分解一下：



### 二次请求

In [9]:
# Prompt
second_template = """
Q: Elsa有5个苹果。Anna比Elsa多两个苹果。她们一共有多少个苹果？
A: 让我们把这个问题分解一下：1.Anna有多少个苹果？2. Elsa和Anna一共有多少个苹果？
1. Anna比Elsa多 2 个苹果。所以Anna有 2 + 5 = 7 个苹果。
2. Elsa和Anna一共有 5 + 7 = 12 个苹果。

Q：{question}
A：让我们把这个问题分解一下：{first_reply}
—–
答案：
"""

second_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", second_template)
    ]
)

# Chain
l2m_second_chain = second_prompt | llm | output_parser

In [10]:
answer = l2m_second_chain.invoke({"question": question, "first_reply": first_reply})
answer

'Albert那天一共吃了48片披萨。'

In [11]:
second_prompt.invoke({"question": question, "first_reply": first_reply})

ChatPromptValue(messages=[SystemMessage(content='\nQ: Elsa有5个苹果。Anna比Elsa多两个苹果。她们一共有多少个苹果？\nA: 让我们把这个问题分解一下：1.Anna有多少个苹果？2. Elsa和Anna一共有多少个苹果？\n1. Anna比Elsa多 2 个苹果。所以Anna有 2 + 5 = 7 个苹果。\n2. Elsa和Anna一共有 5 + 7 = 12 个苹果。\n\nQ：Albert 想知道他一天能吃多少披萨。他买了 2 个大披萨和 2 个小披萨。一个大披萨有 16 片，一个小披萨有 8 片。如果他全部吃掉，那天他吃了多少块？\nA：让我们把这个问题分解一下：1. 两个大披萨一共有 2 * 16 = 32 片。\n2. 两个小披萨一共有 2 * 8 = 16 片。\n3. Albert 一共吃了 32 + 16 = 48 片披萨。\n—–\n答案：\n')])

In [12]:
print(second_prompt.invoke({"question": question, "first_reply": first_reply}).messages[0].content)


Q: Elsa有5个苹果。Anna比Elsa多两个苹果。她们一共有多少个苹果？
A: 让我们把这个问题分解一下：1.Anna有多少个苹果？2. Elsa和Anna一共有多少个苹果？
1. Anna比Elsa多 2 个苹果。所以Anna有 2 + 5 = 7 个苹果。
2. Elsa和Anna一共有 5 + 7 = 12 个苹果。

Q：Albert 想知道他一天能吃多少披萨。他买了 2 个大披萨和 2 个小披萨。一个大披萨有 16 片，一个小披萨有 8 片。如果他全部吃掉，那天他吃了多少块？
A：让我们把这个问题分解一下：1. 两个大披萨一共有 2 * 16 = 32 片。
2. 两个小披萨一共有 2 * 8 = 16 片。
3. Albert 一共吃了 32 + 16 = 48 片披萨。
—–
答案：

